In [1]:
import pandas as pd
import numpy as np

In [3]:
claims = pd.read_csv(r"D:\DEV\Inusrance_fraud_analysis\Files\claims.csv")
claimants = pd.read_csv(r"D:\DEV\Inusrance_fraud_analysis\Files\claimants.csv")
vehicles = pd.read_csv(r"D:\DEV\Inusrance_fraud_analysis\Files\vehicles.csv")
providers = pd.read_csv(r"D:\DEV\Inusrance_fraud_analysis\Files\providers.csv")

In [4]:
print("Claims:", claims.shape)
print("Claimants:", claimants.shape)
print("Vehicles:", vehicles.shape)
print("Providers:", providers.shape)

Claims: (8924, 17)
Claimants: (4000, 15)
Vehicles: (4000, 8)
Providers: (250, 6)


In [5]:
claims.info()

<class 'pandas.DataFrame'>
RangeIndex: 8924 entries, 0 to 8923
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   claim_id              8924 non-null   str    
 1   claimant_id           8924 non-null   str    
 2   vehicle_id            8924 non-null   str    
 3   clinic_provider_id    5151 non-null   str    
 4   repair_provider_id    7677 non-null   str    
 5   attorney_provider_id  1274 non-null   str    
 6   incident_type         8924 non-null   str    
 7   weather_condition     8924 non-null   str    
 8   claim_date            8924 non-null   str    
 9   police_report_filed   8924 non-null   bool   
 10  injuries_count        8924 non-null   int64  
 11  witnesses_count       8924 non-null   int64  
 12  claim_amount          8924 non-null   float64
 13  claim_status          8924 non-null   str    
 14  payout_amount         7874 non-null   float64
 15  is_fraud              8924 non-n

In [8]:
claims.head()

,claim_id,claimant_id,vehicle_id,clinic_provider_id,repair_provider_id,attorney_provider_id,incident_type,weather_condition,claim_date,police_report_filed,injuries_count,witnesses_count,claim_amount,claim_status,payout_amount,is_fraud,fraud_ring_id
0,CM005820,CL002434,VH003054,PR000129,PR000110,NaN,multi_vehicle_pileup,clear,2023-11-28,True,0,0,4946.60,paid,4534.76,0,NaN
1,CM007276,CL002579,VH000249,PR000208,PR000173,PR000215,sideswipe,rain,2023-02-14,True,0,0,1088.40,paid,1019.66,0,NaN
2,CM000287,CL001992,VH000653,NaN,PR000154,PR000237,sideswipe,clear,2023-04-07,True,2,1,3774.64,paid,3730.20,0,NaN
3,CM004253,CL001717,VH000061,PR000156,PR000079,NaN,single_vehicle,rain,2023-04-13,True,1,2,2910.78,paid,2834.66,0,NaN
4,CM008825,CL000486,VH003336,PR000198,PR000086,NaN,parking_lot,clear,2024-08-07,False,1,1,9094.04,denied,0.00,1,RING016


In [9]:
print("Duplicate rows:", claims.duplicated().sum())
print("Duplicated Claim Id:", claims["claim_id"].duplicated().sum())

Duplicate rows: 0
Duplicated Claim Id: 0


In [10]:
print(claims["is_fraud"].value_counts())

is_fraud
0    8400
1     524
Name: count, dtype: int64


In [11]:
claims[claims["payout_amount"].isna()]["claim_status"].value_counts()

claim_status
under_review    1050
Name: count, dtype: int64

In [12]:
print("Claims with fraud ring:")
print(claims["fraud_ring_id"].notna().sum())

print("\nFraud ring distribution:")
print(claims["fraud_ring_id"].value_counts().head(15))

Claims with fraud ring:
524

Fraud ring distribution:
fraud_ring_id
RING001    45
RING010    42
RING018    42
RING008    38
RING004    38
RING005    38
RING003    30
RING000    29
RING015    24
RING011    24
RING012    23
RING009    23
RING019    21
RING007    21
RING016    17
Name: count, dtype: int64


In [13]:
print("Incident types:")
print(claims["incident_type"].value_counts())

print("\nClaim statuses:")
print(claims["claim_status"].value_counts())

print("\nWeather:")
print(claims["weather_condition"].value_counts())

print("\nPolice report:")
print(claims["police_report_filed"].value_counts())

Incident types:
incident_type
single_vehicle          1846
sideswipe               1821
parking_lot             1775
multi_vehicle_pileup    1752
rear_end_collision      1730
Name: count, dtype: int64

Claim statuses:
claim_status
paid              7259
under_review      1050
denied             535
partially_paid      80
Name: count, dtype: int64

Weather:
weather_condition
clear    5443
rain     1728
fog       877
snow      876
Name: count, dtype: int64

Police report:
police_report_filed
True     6779
False    2145
Name: count, dtype: int64


In [14]:
print("Claimant IDs missing from claimants table:",
      (~claims["claimant_id"].isin(claimants["claimant_id"])).sum())

print("Vehicle IDs missing from vehicles table:",
      (~claims["vehicle_id"].isin(vehicles["vehicle_id"])).sum())

print("Repair provider IDs missing:",
      (~claims["repair_provider_id"].dropna().isin(providers["provider_id"])).sum())

print("Clinic provider IDs missing:",
      (~claims["clinic_provider_id"].dropna().isin(providers["provider_id"])).sum())

print("Attorney provider IDs missing:",
      (~claims["attorney_provider_id"].dropna().isin(providers["provider_id"])).sum())

Claimant IDs missing from claimants table: 0
Vehicle IDs missing from vehicles table: 0
Repair provider IDs missing: 0
Clinic provider IDs missing: 0
Attorney provider IDs missing: 0


In [15]:
claims["claim_date"] = pd.to_datetime(
    claims["claim_date"],
    errors="coerce"
)

In [16]:
print("Invalid dates:", claims["claim_date"].isna().sum())

Invalid dates: 0


In [17]:
print("Duplicate claimant IDs:",
      claimants["claimant_id"].duplicated().sum())

print("Duplicate vehicle IDs:",
      vehicles["vehicle_id"].duplicated().sum())

print("Duplicate provider IDs:",
      providers["provider_id"].duplicated().sum())

Duplicate claimant IDs: 0
Duplicate vehicle IDs: 0
Duplicate provider IDs: 0


In [18]:
ring_check = claims[claims["fraud_ring_id"].notna()]["is_fraud"].value_counts()

print(ring_check)

is_fraud
1    524
Name: count, dtype: int64


In [19]:
claims["payout_ratio"] = (
    claims["payout_amount"] / claims["claim_amount"]
)

In [20]:
print(claims["payout_ratio"].isna().sum())

1050


In [21]:
claims["claim_year"] = claims["claim_date"].dt.year
claims["claim_month"] = claims["claim_date"].dt.month

In [22]:
claims["claim_month_name"] = claims["claim_date"].dt.strftime("%b")

In [23]:
claims["unpaid_amount"] = (
    claims["claim_amount"] - claims["payout_amount"]
)

In [24]:
claims[
    ["claim_amount", "payout_amount", "payout_ratio", "unpaid_amount"]
].describe()

,claim_amount,payout_amount,payout_ratio,unpaid_amount
count,8924.000000,7874.000000,7874.000000,7874.000000
mean,4087.768941,3598.946772,0.881270,505.654253
std,2685.865800,2627.133730,0.244120,1340.269293
min,100.410000,0.000000,0.000000,0.130000
25%,2120.852500,1744.412500,0.919414,68.590000
50%,3484.360000,3101.825000,0.946281,165.205000
75%,5414.370000,4931.642500,0.973591,336.020000
max,21878.270000,21128.000000,0.999943,17828.410000


In [25]:
print("Payout ratio > 100%:",
      (claims["payout_ratio"] > 1).sum())

print("Payout ratio < 0%:",
      (claims["payout_ratio"] < 0).sum())

Payout ratio > 100%: 0
Payout ratio < 0%: 0


In [27]:
claims.to_csv(r"D:\DEV\Inusrance_fraud_analysis\Cleaned Data\claims_cleaned.csv", index=False)
claimants.to_csv(r"D:\DEV\Inusrance_fraud_analysis\Cleaned Data\claimants_cleaned.csv", index=False)
vehicles.to_csv(r"D:\DEV\Inusrance_fraud_analysis\Cleaned Data\vehicles_cleaned.csv", index=False)
providers.to_csv(r"D:\DEV\Inusrance_fraud_analysis\Cleaned Data\providers_cleaned.csv", index=False)